# Nextgen AI - SeqGen LSTM (Colab / PyTorch + GPU)

Amaç: **(kullanıcı sorgusu) → (doğal Türkçe yanıt)** ilgisini öğrenen karakter-seviyesi LSTM üretecin GPU ile yeniden eğitimi.

- Eğitim verisi `intents.json` → (pattern, response) çiftleri (artık `seqgen.load_pairs` varsayılanı sorgu koşulludur).
- Model mimarisi `seqgen.SeqModel` ile **birebir aynı** (tek LSTM hücresi, BPTT, kayıp yalnızca yanıt pozisyonlarında).
- Eğitim PyTorch+GPU ile; çıktı NumPy JSON formatında `seq_model.json` olarak kaydedilir → yerel `brain.py` hiç değişmeden çalışır.

## Çalıştırma sırası (sırayla) 
1. Şu dosyayı **`/content`** altına sürükleyip bırakın: `intents.json` 
2. Sırayla tüm hücreleri çalıştırın (Ctrl+F9).
3. Son hücredeki indirme talimatını izleyin.


In [ ]:
import sys, os, io, json, math, time, random
import numpy as np
import torch
sys.path.insert(0, '/content')

from google.colab import drive
USE_DRIVE = False                        # True yaparsan Drive mount + checkpoint istenir
try:
    if USE_DRIVE:
        drive.mount('/content/drive')     # yetki penceresini ONAYLAMALISIN
        DRIVE_DIR = '/content/drive/MyDrive/NextgenAI'
    else:
        DRIVE_DIR = os.path.join(os.getcwd(), 'ckpt')
except Exception as e:
    print('Drive mount yapilamadi, yerel checkpoint kullanilacak:', e)
    DRIVE_DIR = os.path.join(os.getcwd(), 'ckpt')
os.makedirs(DRIVE_DIR, exist_ok=True)

import seqgen
from seqgen import clean_chars, build_vocab, SeqModel, encode_pair, make_batches, save_seq, load_seq

# ---------------- hiperparametreler
HIDDEN     = 256     # LSTM gizli boyut (numpy ile ayni cebir)
BATCH_SIZE = 128
EPOCHS     = 250
LR_BASE    = 2e-3
LR_MIN     = 0.01    # cosine decay alt siniri
PATIENCE   = 15      # erken durdurma
GRAD_CLIP  = 5.0
CKPT_FREQ  = 3       # kac epoch'ta bir Drive checkpoint
MAX_PAIRS  = 20000
SEED       = 7

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', DEVICE, '| GPU:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '-')


In [ ]:
# intents.json -> (sorgu, yanit) ciftleri; 10% val ayrimi (seed 7).
INTENTS = '/content/intents.json'
assert os.path.exists(INTENTS), 'intents.json yuklenmedi! Sol panelden /content icine surukleyin.'

pairs = seqgen.load_pairs(INTENTS, max_pairs=MAX_PAIRS, use_query=True)
print('egitim cifti (sorgu, yanit):', len(pairs))
if len(pairs) < 3000:
    print('UYARI: az cift; zengin intents.json onerilir.')

all_text = []
for ctx, resp in pairs:
    all_text.append(ctx); all_text.append(resp)
vocab = build_vocab(all_text)
print('karakter sozlugu:', len(vocab))

dummy = SeqModel(vocab, hidden=4)               # sadece c2i eslemesi
rng = np.random.RandomState(SEED)
perm = rng.permutation(len(pairs))
n_val = max(1, int(0.1 * len(pairs)))
tr_pairs = [pairs[i] for i in perm[n_val:]]
va_pairs = [pairs[i] for i in perm[:n_val]]
tr = make_batches(dummy, tr_pairs, BATCH_SIZE)
va = make_batches(dummy, va_pairs, BATCH_SIZE)
print('train batch:', len(tr), '| val batch:', len(va))
print('ornek cift:', tr_pairs[0])


In [ ]:
class TorchCharLSTM(torch.nn.Module):
    """seqgen.SeqModel ile birebir AYNI LSTM cebiri (PyTorch).
    Ek: drop (egitimde Wy oncesi h'ya; eval'da kapali -> parity bozulmaz)."""
    def __init__(self, vocab_size, hidden, bound=0.08, drop=0.15):
        super().__init__()
        self.V, self.H = vocab_size, hidden
        self.drop = drop
        self.Wxh = torch.nn.Parameter(torch.zeros(vocab_size, 4*hidden).uniform_(-bound, bound))
        self.Whh = torch.nn.Parameter(torch.zeros(hidden, 4*hidden).uniform_(-bound, bound))
        self.bh  = torch.nn.Parameter(torch.zeros(1, 4*hidden))
        self.Wy  = torch.nn.Parameter(torch.zeros(hidden, vocab_size).uniform_(-bound, bound))
        self.by  = torch.nn.Parameter(torch.zeros(1, vocab_size))

    def forward(self, inp):
        B, L = inp.shape
        one = torch.nn.functional.one_hot(inp, self.V).float()
        H = self.H
        h = inp.new_zeros(B, H).float(); c = inp.new_zeros(B, H).float()
        outs = []
        for s in range(L):
            a = one[:, s] @ self.Wxh + h @ self.Whh + self.bh
            i = torch.sigmoid(a[:, 0:H]); f = torch.sigmoid(a[:, H:2*H])
            g = torch.tanh(a[:, 2*H:3*H]); o = torch.sigmoid(a[:, 3*H:4*H])
            c = f*c + i*g; h = o*torch.tanh(c)
            y = torch.dropout(h, self.drop, train=self.training)
            outs.append(y @ self.Wy + self.by)
        return torch.stack(outs, 1)          # (B, L, V)


In [ ]:
def seq_loss(logits, tgt, mask):
    """Kayip yalnizca maskeli (yanit) pozisyonlarda; numpy forward ile ayni."""
    lg = torch.log_softmax(logits, dim=-1)
    nll = lg.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
    return -(nll * mask).sum() / mask.sum().clamp(min=1.0)

def masked_acc(logits, tgt, mask):
    ok = ((logits.argmax(-1) == tgt) & mask.bool())
    return ok.sum().item() / mask.sum().clamp(min=1.0).item()


In [ ]:
trX = [torch.from_numpy(b[0]).long().to(DEVICE) for b in tr]
trT = [torch.from_numpy(b[1]).long().to(DEVICE) for b in tr]
trM = [torch.from_numpy(b[2]).float().to(DEVICE) for b in tr]
vaX = [torch.from_numpy(b[0]).long().to(DEVICE) for b in va]
vaT = [torch.from_numpy(b[1]).long().to(DEVICE) for b in va]
vaM = [torch.from_numpy(b[2]).float().to(DEVICE) for b in va]

CKPT = os.path.join(DRIVE_DIR, 'seqgen_ckpt.pt')

model = TorchCharLSTM(len(vocab), HIDDEN).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR_BASE)

best_state = None
best_val = 1e9
bad = 0
start_ep = 0
tot_steps = EPOCHS * len(trX)

if os.path.exists(CKPT):                      # kesinti sonrasi kaldigi yerden
    cp = torch.load(CKPT, map_location=DEVICE, weights_only=True)
    model.load_state_dict(cp['model']); opt.load_state_dict(cp['opt'])
    best_val, best_state, start_ep = cp['best_val'], cp['best_state'], cp['epoch']
    best_state = {k: v.detach().cpu().clone() for k, v in best_state.items()}
    print('Devam: epoch', start_ep, '| best val:', round(best_val, 4))


In [ ]:
t0_all = time.time()
done = False
for ep in range(start_ep + 1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    order = list(range(len(trX))); random.Random(ep).shuffle(order)
    step_g = (ep - 1) * len(trX)
    tl = 0.0
    for bi in order:
        prog = (step_g + 1) / tot_steps
        lr = LR_MIN + 0.5 * (LR_BASE - LR_MIN) * (1 + math.cos(math.pi * prog))
        for g in opt.param_groups:
            g['lr'] = lr
        opt.zero_grad()
        loss = seq_loss(model(trX[bi]), trT[bi], trM[bi])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()
        step_g += 1
        tl += loss.item()
    tl /= len(trX)

    model.eval(); vl = va_acc = 0.0
    with torch.no_grad():
        for x, t, m in zip(vaX, vaT, vaM):
            lg = model(x)
            vl += seq_loss(lg, t, m).item()
            va_acc += masked_acc(lg, t, m)
    vl /= len(vaX); va_acc /= len(vaX)
    print(f'epoch {ep:3d}/{EPOCHS} | train {tl:.4f} | val {vl:.4f} | acc {va_acc:.3f} | {time.time()-t0:.1f}s | lr {lr:.4f}', flush=True)

    if vl < best_val - 1e-4:
        best_val = vl
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1
        if bad >= PATIENCE:
            print(f'[seqgen] Erken durdurma. Best val: {best_val:.4f}')
            done = True
    if ep % CKPT_FREQ == 0 or done:
        torch.save({'epoch': ep, 'model': best_state, 'opt': opt.state_dict(),
                    'best_val': best_val, 'best_state': best_state}, CKPT)
        print(f'  checkpoint -> {CKPT}')
    if done:
        break

print('\nToplam egitim suresi: %.1f dk' % ((time.time() - t0_all) / 60))


In [ ]:
model.load_state_dict(best_state); model.eval()

best_state = {k: v.detach().cpu().clone() for k, v in best_state.items()}

m = SeqModel(vocab, hidden=HIDDEN)
for k in ['Wxh', 'Whh', 'bh', 'Wy', 'by']:
    setattr(m, k, best_state[k].numpy())

out_dir = '/content/model'
os.makedirs(out_dir, exist_ok=True)
save_seq(m, os.path.join(out_dir, 'seq_model.json'))
save_seq(m, os.path.join(DRIVE_DIR, 'seq_model.json'))       # Drive yedegi
print('model/seq_model.json yazildi (indirmeye hazir):', os.path.join(out_dir, 'seq_model.json'))


In [ ]:
# Dipnoh: PyTorch egitimi ile yereldeki numpy SeqModel BIREBIR ayni mi?
loaded = load_seq(os.path.join(out_dir, 'seq_model.json'))
for k in ['Wxh', 'Whh', 'bh', 'Wy', 'by']:
    d = float(torch.abs(best_state[k] - torch.as_tensor(getattr(loaded, k))).max())
    print(f'{k}: diff {d:.3e}')

inp, tgt, mask = tr[0]
with torch.no_grad():
    lt = seq_loss(model(torch.from_numpy(inp).long().to(DEVICE)),
                  torch.from_numpy(tgt).long().to(DEVICE),
                  torch.from_numpy(mask).float().to(DEVICE)).item()
lnp, _, _, _, _ = loaded.forward(inp, tgt, mask)
print('torch loss:', lt, '| numpy loss:', lnp, '| diff:', abs(lt - lnp))
assert abs(lt - lnp) < 1e-3, 'PARITY FAIL'
print('PARITY OK')


In [ ]:
print('=== SEQGEN ORNEKLERI (temperature 0.9, top_k 14) ===')
for q in ['hava nasil olacak', 'yapay zeka nedir', 'tesekkur ederim',
          'en iyi film hangisi', 'kedi bakimi nasil olur', 'okul ne zaman kapanir']:
    print(f'? {q}\nAI: {loaded.sample(q, temperature=0.9, top_k=14)}\n')


In [ ]:
from google.colab import files
import shutil
print('Model dosyasi:', os.path.join(out_dir, 'seq_model.json'))
# Dosyayi indirmek icin:  files.download('/content/model/seq_model.json')


## İndirme & yerel kurulum

1. `model/seq_model.json` (üst hücredeki buton veya `files.download(...)`) indir.
2. Dosyayı projenin `model/` klasörüne kopyala: `Nextgen_API/model/seq_model.json`
3. Yerel test:

```python
from seqgen import load_seq
m = load_seq()
print(m.sample('hava nasil olacak', temperature=0.9, top_k=14))
```

4. `brain.py` seqgen'i artık kullanıcı **sorgusuyla** örnekliyor; `seq_model.json` varsa otomatik açılır, kalite kapısından geçemeyen çıktı canned cevaba düşer.
